# 📝 EDA·데이터 시각화 과제 LV3 정답 — 데이터셋 EDA 리포트 (강사용)

각 단계의 **모범 코드 + 자가채점 + 해설** 입니다.

In [ ]:
# [제공 코드] 시각화 라이브러리와 한글 폰트를 준비합니다.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 **어떤 데이터인지 먼저 파악**합니다. `head()` 로 앞부분을, `info()` 로 열·자료형·결측을, `describe()` 로 수치 요약을 봅니다. (아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·수치 요약
df = pd.read_csv('../../day07_EDA_시각화/data/diamonds.csv')
print("행·열 크기:", df.shape)
print("\n[앞 5행] head()"); display(df.head())
print("\n[열·자료형·결측] info()"); df.info()
print("\n[수치 요약] describe()"); display(df.describe())

## 1. 다이아몬드 컷·색 품질 EDA 리포트
**배경**: `diamonds.csv` 는 다이아몬드 5000개의 무게·컷·색·가격입니다. 이번에는 **컷(cut)·색(color) 같은 품질 등급**이 어떻게 분포하고, 그 등급이 가격과 어떻게 이어지는지 집계·시각화로 탐색합니다.

### 1단계 — 데이터 불러오기·구조 파악
`data/diamonds.csv` 를 `df` 로 불러오고, `df.shape` 와 `df['cut'].value_counts()`, `df['color'].value_counts()`, 그리고 `df['price'].describe()` 를 출력하세요. 원본은 `(5000, 10)` 이고, `cut` 은 5종(Fair·Good·Very Good·Premium·Ideal), `color` 는 7종(D~J) 입니다.

In [ ]:
df = pd.read_csv('../../day07_EDA_시각화/data/diamonds.csv')
print(df.shape)
print(df['cut'].value_counts())
print(df['color'].value_counts())
print(df['price'].describe())

In [ ]:
# [자가채점]
assert df.shape == (5000, 10)
assert df['cut'].nunique() == 5
assert df['color'].nunique() == 7
print("✅ 1단계 통과!")

### 2단계 — 집계 (groupby · pivot_table · crosstab · 사용자 정의 집계)
같은 `df` 로 세 가지 품질 집계를 만드세요.
- `price_by_color`: `color` 별 평균 `price`(색 등급별 평균 가격) — `groupby`
- `pv`: `cut`(행) × `color`(열) 의 평균 `carat`(평균 무게) — `pivot_table` (모양은 `(5, 7)`)
- `ct`: `cut`(행) × `color`(열) 의 **개수** 교차표 — `crosstab` (모양은 `(5, 7)`)
- `color_summary`: `color` 별로 아래 **세 열**을 가진 표 — **사용자 정의 집계**(결과 열 이름을 직접 붙이고, 소수 둘째 자리로 반올림)
  - `평균가` — `price` 의 평균  ·  `가격범위` — `price` 의 **최댓값 − 최솟값**(`lambda`)  ·  `고가비율` — `price` 가 **10000 이상**인 비율(%)  ·  `최빈_투명도` — `clarity` 의 **최빈값**(`mode()[0]`)
  - 앞의 셋은 수치형 집계, 마지막 하나는 **범주형 집계**입니다. 색 등급이 좋으면 투명도 등급도 좋은지 함께 보려는 것이죠.

(참고: D색 평균가 2999.74, J색 평균가 5755.88, Ideal·G 평균 캐럿 0.666, Ideal·G 개수 437, D색 가격범위 18202, J색 고가비율 18.73, D색 최빈 투명도 SI1, G색 최빈 투명도 VS2)

In [ ]:
price_by_color = df.groupby('color')['price'].mean()
pv = df.pivot_table(index='cut', columns='color', values='carat')
ct = pd.crosstab(df['cut'], df['color'])
color_summary = df.groupby('color').agg(
    평균가=('price', 'mean'),
    가격범위=('price', lambda s: s.max() - s.min()),
    고가비율=('price', lambda s: (s >= 10000).mean() * 100),
    최빈_투명도=('clarity', lambda s: s.mode()[0]),      # 범주형 -> 최빈값
).round(2)
print(price_by_color.round(2).to_dict())
print(pv.round(3))
print(ct)

In [ ]:
# [자가채점]
assert round(float(price_by_color['D']), 2) == 2999.74
assert round(float(price_by_color['J']), 2) == 5755.88
assert pv.shape == (5, 7)
assert round(float(pv.loc['Ideal', 'G']), 3) == 0.666
assert ct.shape == (5, 7)
assert int(ct.loc['Ideal', 'G']) == 437
assert color_summary.shape == (7, 4), '색 7종 × 4열이 나와야 해요'
assert list(color_summary.columns) == ['평균가', '가격범위', '고가비율', '최빈_투명도'], '열 이름을 지문 그대로 써 주세요'
assert round(float(color_summary.loc['D', '평균가']), 2) == 2999.74
assert round(float(color_summary.loc['D', '가격범위']), 2) == 18202.0
assert color_summary.loc['D', '최빈_투명도'] == 'SI1', 'clarity 의 최빈값이에요'
assert color_summary.loc['G', '최빈_투명도'] == 'VS2'
assert round(float(color_summary.loc['J', '고가비율']), 2) == 18.73, '10000 이상인 비율(%)이에요 — 100을 곱했나요?'
print("✅ 2단계 통과!")

### 3단계 — 시각화 3종 (범주·분포·관계)
같은 `df` 로 세 그래프를 그리고, 각 결과 Axes 를 지정 이름에 저장한 뒤 제목을 다세요.
- `ax1`: **컷 등급별 개수 막대그래프** — `countplot(x='cut')` (범주)
- `ax2`: **무게(carat) 분포 히스토그램** — `histplot(x='carat', bins=30)` (분포)
- `ax3`: **무게-가격 산점도(색 등급별)** — `scatterplot(x='carat', y='price', hue='color')` (관계)
- 그래프가 겹치지 않도록 각 그래프 앞에 `plt.figure()` 를 호출하세요.

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day07_EDA_시각화/images/과제/lv3_q1_s3_1.png" width="520"/>

<img src="../../day07_EDA_시각화/images/과제/lv3_q1_s3_2.png" width="520"/>

<img src="../../day07_EDA_시각화/images/과제/lv3_q1_s3_3.png" width="520"/>

In [ ]:
plt.figure()
ax1 = sns.countplot(data=df, x='cut')
ax1.set_title('컷 등급별 개수')
plt.show()

plt.figure()
ax2 = sns.histplot(data=df, x='carat', bins=30)
ax2.set_title('무게(캐럿) 분포')
plt.show()

plt.figure()
ax3 = sns.scatterplot(data=df, x='carat', y='price', hue='color')
ax3.set_title('무게와 가격 (색 등급별)')
plt.show()

### 4단계 — 인사이트 (서술)
위 집계와 그래프를 근거로 **컷·색 품질 등급이 가격과 어떻게 이어지는지**를 **3문장 이상** 서술하세요.
- 그래프에서 **보이는 경향**을 말로 표현하세요 (예: 어떤 등급이 더 흔한지, 무게가 클수록 가격이 어떻게 되는지).

**인사이트 (모범 서술)**

품질 등급은 시장에서 고르게 퍼져 있지 않습니다. countplot 과 개수 교차표를 보면 컷은 Ideal(1953개)이 압도적으로 많고 Fair(162개)는 드물며, 그 안에서도 Ideal·G 조합(437개)처럼 특정 등급 조합에 물량이 몰려 있습니다. 흥미롭게도 색 등급별 평균 가격은 가장 맑은 D색이 2999로 가장 낮고 가장 누런 J색이 5756으로 가장 높은데, 이는 평균 캐럿 표에서 보이듯 높은 등급일수록 작은 돌이 많아(예: Ideal·G 0.666캐럿) 무게 효과가 등급 효과를 덮기 때문으로 읽을 수 있습니다. 사용자 정의 집계로 만든 색 등급별 요약표도 같은 이야기를 합니다 — 가격범위는 어느 색이나 18000 안팎으로 비슷하게 넓지만(D색 18202), 1만 달러가 넘는 고가품의 비율은 D색 6.14%에서 J색 18.73%로 세 배 넘게 벌어집니다. 즉 색 등급이 낮은 쪽에 비싼 돌이 몰려 있다는 뜻이고, 이는 그쪽에 큰 돌이 많다는 앞의 해석과 맞물립니다. 한편 색 등급별 최빈 투명도를 보면 최고 색인 D가 오히려 SI1(중간 아래)이고 G는 VS2로 더 높아, **색 등급과 투명도 등급은 함께 움직이지 않습니다** — 품질 등급을 하나로 뭉뚱그려 말하면 안 되는 이유입니다. 산점도에서도 무게(carat)가 커질수록 가격이 뚜렷하게 위로 올라가, 품질 등급보다 무게가 가격을 더 크게 끌어올리는 경향을 눈으로 확인할 수 있습니다.

## 2. 다이아몬드 가격 EDA 리포트
**배경**: `diamonds.csv` 는 다이아몬드 5000개의 무게·컷·색·가격입니다. 가격이 무엇에 좌우되는지 집계·시각화로 탐색합니다.

### 1단계 — 데이터 불러오기·기초 파악
`data/diamonds.csv` 를 `df` 로 불러오고, `df.shape` 와 함께 `df['carat'].describe()`, `df['price'].describe()` 를 출력하세요. 원본은 `(5000, 10)`, `price` 평균은 약 3917.29, `carat` 평균은 약 0.797 입니다.

In [ ]:
df = pd.read_csv('../../day07_EDA_시각화/data/diamonds.csv')
print(df.shape)
print(df['carat'].describe())
print(df['price'].describe())

In [ ]:
# [자가채점]
assert df.shape == (5000, 10)
assert round(float(df['price'].mean()), 2) == 3917.29
assert round(float(df['carat'].mean()), 3) == 0.797
print("✅ 1단계 통과!")

### 2단계 — 집계 (groupby · pivot_table · apply)
같은 `df` 로 두 집계를 만드세요.
- `price_by_cut`: `cut` 별 평균 `price` — `groupby`
- `pv`: `cut`(행) × `color`(열) 의 평균 `price` — `pivot_table` (모양은 `(5, 7)`)
- `ppc_by_cut`: `cut` 별 **'캐럿당 가격'의 평균** — 다이아 **한 개씩** `price ÷ carat` 을 구한 뒤 그 평균을 냅니다. 두 열을 함께 써야 하니 `apply` 로 만드세요(결과는 Series, 인덱스는 `cut`). 쓸 두 열만 골라 넘기면(`[['price', 'carat']]`) 경고 없이 깔끔합니다.
  - ⚠️ `price` **합계** ÷ `carat` **합계**로 계산하면 값이 달라집니다(Fair 3984.43). **행마다 나눈 뒤 평균**을 내세요.

(참고: Premium 평균가 4587.82, Ideal 평균가 3321.98, Ideal·G 조합 3387.32, 캐럿당 가격 Fair 3550.40 · Premium 4201.88)

In [ ]:
price_by_cut = df.groupby('cut')['price'].mean()
pv = df.pivot_table(index='cut', columns='color', values='price')
ppc_by_cut = df.groupby('cut')[['price', 'carat']].apply(
    lambda g: (g['price'] / g['carat']).mean())
print(price_by_cut.round(2).to_dict())
print(pv.round(1))

In [ ]:
# [자가채점]
assert round(float(price_by_cut['Premium']), 2) == 4587.82
assert round(float(price_by_cut['Ideal']), 2) == 3321.98
assert pv.shape == (5, 7)
assert round(float(pv.loc['Ideal', 'G']), 2) == 3387.32
assert round(float(ppc_by_cut['Fair']), 2) == 3550.4, '행마다 price/carat 을 구한 뒤 평균을 내세요(합계로 나누면 3984.43 이 나옵니다)'
assert round(float(ppc_by_cut['Premium']), 2) == 4201.88
print("✅ 2단계 통과!")

### 3단계 — 시각화 3종 (분포·관계·집계 히트맵)
같은 `df`(와 2단계의 `pv`)로 세 그래프를 그리고, 각 결과 Axes 를 지정 이름에 저장한 뒤 제목을 다세요.
- `ax1`: **가격 분포 히스토그램** — `histplot(x='price', bins=30)` (분포)
- `ax2`: **무게-가격 산점도** — `scatterplot(x='carat', y='price', hue='cut')` (관계)
- `ax3`: **컷×색 평균가격 히트맵** — 2단계 `pv` 를 `heatmap(annot=True, fmt='.0f')` 로 (집계값 히트맵)
- 그래프가 겹치지 않도록 각 그래프 앞에 `plt.figure()` 를 호출하세요.

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day07_EDA_시각화/images/과제/lv3_q2_s3_1.png" width="520"/>

<img src="../../day07_EDA_시각화/images/과제/lv3_q2_s3_2.png" width="520"/>

<img src="../../day07_EDA_시각화/images/과제/lv3_q2_s3_3.png" width="520"/>

In [ ]:
plt.figure()
ax1 = sns.histplot(data=df, x='price', bins=30)
ax1.set_title('가격 분포')
plt.show()

plt.figure()
ax2 = sns.scatterplot(data=df, x='carat', y='price', hue='cut')
ax2.set_title('무게와 가격 (컷 등급별)')
plt.show()

plt.figure()
ax3 = sns.heatmap(pv, annot=True, fmt='.0f')
ax3.set_title('컷×색 평균 가격')
plt.show()

### 4단계 — 인사이트 (서술)
위 집계와 그래프를 근거로 **가격을 좌우하는 요인**을 **3문장 이상** 서술하세요.
- 그래프에서 **보이는 경향**을 말로 표현하세요 (예: 캐럿이 클수록 가격이 오르는 편).

**인사이트 (모범 서술)**

가격을 가장 크게 좌우하는 것은 **무게(carat)** 로 보입니다. 산점도에서 무게가 커질수록 가격이 뚜렷하게 위로 올라가는 경향이 나타납니다. 흥미롭게도 컷 등급별 평균가는 최고 등급인 Ideal(3322)이 오히려 낮고 Premium(4588)이 높은데, 이는 Ideal 다이아몬드에 작은 것이 많이 섞여 있어 무게 효과가 컷 효과를 덮기 때문으로 읽을 수 있습니다. 이 해석의 결정적 근거가 `ppc_by_cut` 입니다 — 무게로 나눠 크기 효과를 걷어 내면 Ideal 은 캐럿당 3825달러로 Good(3813)·Fair(3550)보다 **높아집니다**. 평균가에서 최저였던 Ideal 이 크기를 보정하면 뒤집히는 것이죠. 평균만 보고 '컷이 좋아도 싸다'고 결론 내리면 안 되는 이유입니다. 히트맵을 보면 같은 컷 안에서도 색 등급에 따라 평균가가 달라져, 가격은 무게를 중심으로 컷·색이 함께 작용함을 알 수 있습니다.